# Scaling Law Visualization

Visualize results from the scaling law experiments.

Run this after `scaling_law_colab.ipynb` or `python -m scripts.run_scaling_experiment`.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULTS_DIR = "results_scaling_laws"
csv_path = os.path.join(RESULTS_DIR, "scaling_results.csv")

df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} experiment runs from {csv_path}")

summary = df.groupby("multiplier").agg(
    qlike_mean=("qlike", "mean"),
    qlike_std=("qlike", "std"),
    mse_mean=("mse", "mean"),
    mse_std=("mse", "std"),
    mae_mean=("mae", "mean"),
    mae_std=("mae", "std"),
    n_windows=("n_train_windows", "first"),
).reset_index()
print(summary.to_string(index=False))

In [ ]:
# --- QLIKE vs multiplier and vs training set size ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].errorbar(
    summary["multiplier"], summary["qlike_mean"], yerr=summary["qlike_std"],
    fmt="o-", capsize=4, color="#1f77b4",
)
axes[0].set_xlabel("Synthetic Data Multiplier")
axes[0].set_ylabel("QLIKE")
axes[0].set_title("Scaling Law: QLIKE vs Data Augmentation")
axes[0].grid(True, alpha=0.3)

axes[1].errorbar(
    summary["n_windows"], summary["qlike_mean"], yerr=summary["qlike_std"],
    fmt="s-", capsize=4, color="#d62728",
)
axes[1].set_xlabel("Total Training Windows")
axes[1].set_ylabel("QLIKE")
axes[1].set_title("Scaling Law: QLIKE vs Training Set Size")
axes[1].set_xscale("log")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- All metrics scaling curves ---
x = summary["multiplier"].values
x_plot = np.where(x == 0, 0.5, x).astype(float)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, metric, label in zip(
    axes,
    ["qlike", "mse", "mae"],
    ["QLIKE", "MSE (adj scale)", "MAE (adj scale)"],
    strict=False,
):
    ax.errorbar(
        x_plot,
        summary[f"{metric}_mean"],
        yerr=summary[f"{metric}_std"],
        fmt="o-",
        capsize=4,
        linewidth=2,
        markersize=8,
        color="#1f77b4",
    )
    ax.set_xscale("log")
    ax.set_xticks(x_plot)
    ax.set_xticklabels(
        [f"{int(m)}x" if m > 0 else "0" for m in x],
        fontsize=10,
    )
    ax.get_xaxis().set_major_formatter(plt.FuncFormatter(lambda val, pos: ""))
    ax.set_xlabel("Synthetic Multiplier", fontsize=11)
    ax.set_ylabel(label, fontsize=11)
    ax.set_title(label, fontsize=12, fontweight="bold")
    ax.grid(True, alpha=0.3, which="both")

fig.suptitle("Scaling Laws: All Metrics", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()